In [1]:
import pandas as pd
from utils import ArticleInput, AnalysisResult, AnalysisOptions, SentenceScore

class CheckworthyExperiment:
    def __init__(self, classifier):
        self.classifier = classifier
        # Strategy Fix: More specific labels for news data
        self.candidate_labels = [
            "factual news reporting", 
            "personal opinion", 
            "interrogative question", 
            "navigation or spam"
        ]

    def test_logic(self, sentence_text, centrality_score, threshold=0.60):
        """
        Runs the hybrid logic to show how Gates 2 & 3 override model errors.
        """
        # Run the actual Zero-Shot Model
        pred = self.classifier(sentence_text, self.candidate_labels, multi_label=False)
        top_label = pred['labels'][0]
        top_score = pred['scores'][0]

        # --- THE THREE GATES ---
        # Gate 1: Semantic
        is_factual = (top_label == "factual news reporting" and top_score >= threshold)
        
        # Gate 2: Numerical Heuristic
        has_numbers = any(char.isdigit() for char in sentence_text)
        
        # Gate 3: Centrality Guard
        is_central = centrality_score > 0.75

        # Final Determination
        is_checkworthy = is_factual or (is_central and has_numbers)
        
        reason = "Semantic Match" if is_factual else "Heuristic Override" if (is_central and has_numbers) else "Rejected"

        return {
            "text": sentence_text[:60] + "...",
            "model_label": top_label,
            "model_conf": f"{top_score:.2f}",
            "has_numbers": has_numbers,
            "is_central": is_central,
            "FINAL_DECISION": "✅ KEEP" if is_checkworthy else "❌ DISCARD",
            "reason": reason
        }

# --- RUNNING THE EXPERIMENT ---

# Use the sentence that failed in your previous test_output
test_sentences = [
    ("Hundreds of photos... leaked to BBC Verify", 0.98), # High centrality
    ("The pictures reveal at least 326 victims - including 18 women.", 0.76), # Numbers + Centrality
    ("One source, who we are not naming for their safety, told us...", 0.40), # Low centrality
]

lab = CheckworthyExperiment(classifier) # Assuming classifier is loaded in previous cell
results = [lab.test_logic(text, score) for text, score in test_sentences]

# Display as a Table for easy comparison
pd.DataFrame(results)

NameError: name 'classifier' is not defined